# Лабораторная работа №3. Решающее дерево

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются те же Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE), что в [лабораторной №1](lab1.ipynb). Тест и фолды не меняются. Основные метрики подбора: macro-F1 вверх, MAE вниз.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует исходные числовые признаки.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'raw'

def sklearn_model(task, params):
    if task == 'classification':
        return DecisionTreeClassifier(random_state=SEED, **params)
    return DecisionTreeRegressor(random_state=SEED, **params)

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [ ]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

## Гипотезы и кросс-валидация

Дерево без ограничений может подогнаться под редких покупателей и отдельные замеры. Проверяю ограничения глубины и минимального размера листа. Отдельно проверяю доли расходов и составные признаки плитки: одиночные монотонные логарифмы сами по себе почти не меняют порядок порогов.

В обоих случаях 5 фолдов. Сравниваю препроцессинг без подбора, подбор без изменения признаков и их комбинацию. Сначала выбираю лучшую альтернативу бейзлайну на CV, затем оцениваю её на отложенной части. `cv_std` показывает разброс оценок по фолдам, это не доверительный интервал.

In [ ]:
feature_options = ['features']
class_grid = [{'max_depth': 3, 'min_samples_leaf': 2}, {'max_depth': 3, 'min_samples_leaf': 5}, {'max_depth': 3, 'min_samples_leaf': 10}, {'max_depth': 6, 'min_samples_leaf': 2}, {'max_depth': 6, 'min_samples_leaf': 5}, {'max_depth': 6, 'min_samples_leaf': 10}, {'max_depth': 10, 'min_samples_leaf': 2}, {'max_depth': 10, 'min_samples_leaf': 5}, {'max_depth': 10, 'min_samples_leaf': 10}]
reg_grid = [{'max_depth': 3, 'min_samples_leaf': 2}, {'max_depth': 3, 'min_samples_leaf': 5}, {'max_depth': 3, 'min_samples_leaf': 10}, {'max_depth': 6, 'min_samples_leaf': 2}, {'max_depth': 6, 'min_samples_leaf': 5}, {'max_depth': 6, 'min_samples_leaf': 10}, {'max_depth': 10, 'min_samples_leaf': 2}, {'max_depth': 10, 'min_samples_leaf': 5}, {'max_depth': 10, 'min_samples_leaf': 10}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

Лучший вариант из альтернатив заново обучаю на всём train. Прирост на CV не гарантирует прироста на тесте.

In [ ]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))